In [109]:
import numpy as np
import pandas as pd
import random

df = pd.read_csv('../data/hotel_bookings.csv', usecols=['hotel', 'is_canceled', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'stays_in_weekend_nights',
                                                        'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'reservation_status', 'reservation_status_date'])
df.head()
df.shape

(119390, 13)

In [110]:
#filter to resort hotels and confirmed checkouts only

df = df[(df['hotel'] == 'Resort Hotel') & (df['reservation_status'] == 'Check-Out') & (df['is_canceled'] == 0)].copy()

pd.set_option('display.max_colwidth', None)  # show full text in cells
pd.set_option('display.width', None) 

#build arrival date from the day, month, and year columns
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + ' '
    + df['arrival_date_month'].astype(str) + ' '
        + df['arrival_date_day_of_month'].astype(str),
        format='%Y %B %d') 

#total guests per booking

df['total_guests'] = df['adults'] + df['children'].fillna(0) + df['babies'].fillna(0)

#length of stay in nights

df['length_of_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']


df['depature_date'] = df['arrival_date'] + pd.to_timedelta(df['length_of_stay'], unit='D')

#get rid of all the guests who did not stay a night which means they probably did not need the restaurant for breakfast 
df = df[(df['length_of_stay'] > 0)].copy()


df['breakfast_included'] = df['meal'].isin(['BB', 'HB', 'FB'])

df['half_board'] = df['meal'].isin(['HB'])

df['full_board'] = df['meal'].isin(['FB'])

df.head(500)



,hotel,is_canceled,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,reservation_status,reservation_status_date,arrival_date,total_guests,length_of_stay,depature_date,breakfast_included,half_board,full_board
2,Resort Hotel,0,2015,July,1,0,1,1,0.0,0,BB,Check-Out,2015-07-02,2015-07-01,1.0,1,2015-07-02,True,False,False
3,Resort Hotel,0,2015,July,1,0,1,1,0.0,0,BB,Check-Out,2015-07-02,2015-07-01,1.0,1,2015-07-02,True,False,False
4,Resort Hotel,0,2015,July,1,0,2,2,0.0,0,BB,Check-Out,2015-07-03,2015-07-01,2.0,2,2015-07-03,True,False,False
5,Resort Hotel,0,2015,July,1,0,2,2,0.0,0,BB,Check-Out,2015-07-03,2015-07-01,2.0,2,2015-07-03,True,False,False
6,Resort Hotel,0,2015,July,1,0,2,2,0.0,0,BB,Check-Out,2015-07-03,2015-07-01,2.0,2,2015-07-03,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
669,Resort Hotel,0,2015,July,25,0,1,2,0.0,0,BB,Check-Out,2015-07-26,2015-07-25,2.0,1,2015-07-26,True,False,False
671,Resort Hotel,0,2015,July,25,1,1,2,0.0,0,BB,Check-Out,2015-07-27,2015-07-25,2.0,2,2015-07-27,True,False,False
675,Resort Hotel,0,2015,July,25,1,1,2,0.0,0,BB,Check-Out,2015-07-27,2015-07-25,2.0,2,2015-07-27,True,False,False
676,Resort Hotel,0,2015,July,25,2,1,2,0.0,0,HB,Check-Out,2015-07-28,2015-07-25,2.0,3,2015-07-28,True,True,False


In [111]:
#  one row -> one booking - the arrival and depature date and the total number of people  on that booking.
# We need to expand each booking across every night of the stay then aggregate per day to get the number of people staying at the hotel
# at a time.

def expand_booking(row):
    dates = pd.date_range(start=row['arrival_date'], end=row['depature_date'], inclusive='left')
    return pd.DataFrame({
        'date': dates,
        'total_guests': row['total_guests'],
        'meal': row['meal'],
        'breakfast_included': row['breakfast_included'],
        'half_board': row['half_board'],
        'full_board': row['full_board']
    })


expanded = pd.concat(
    [expand_booking(row) for _, row in df.iterrows()], ignore_index=True
)

print(f"Expanded shape: {expanded.shape}")
expanded.head(10)

Expanded shape: (119887, 6)


,date,total_guests,meal,breakfast_included,half_board,full_board
0,2015-07-01,1.0,BB,True,False,False
1,2015-07-01,1.0,BB,True,False,False
2,2015-07-01,2.0,BB,True,False,False
3,2015-07-02,2.0,BB,True,False,False
4,2015-07-01,2.0,BB,True,False,False
5,2015-07-02,2.0,BB,True,False,False
6,2015-07-01,2.0,BB,True,False,False
7,2015-07-02,2.0,BB,True,False,False
8,2015-07-01,2.0,FB,True,False,True
9,2015-07-02,2.0,FB,True,False,True


In [112]:
#aggregate per calendar date

daily = (
    expanded.assign(
        guests_with_breakfast = lambda x: x['total_guests'] * x['breakfast_included'],
        guests_half_board = lambda x: x['total_guests'] * x['half_board'],
        guests_full_board = lambda x: x['total_guests'] * x['full_board']
    )
    .groupby('date', as_index=False)
    .agg(
        guests_in_hotel = ('total_guests', 'sum'),
        guests_with_breakfast = ('breakfast_included', 'sum'),
        guests_half_board = ('half_board', 'sum'),
        guests_full_board = ('full_board', 'sum'),
    )
    .sort_values('date')
    .reset_index(drop=True)
)

print(f"Daily shape: {daily.shape}")
daily.head(10)


Daily shape: (806, 5)


,date,guests_in_hotel,guests_with_breakfast,guests_half_board,guests_full_board
0,2015-07-01,68.0,36,4,1
1,2015-07-02,131.0,64,11,2
2,2015-07-03,168.0,81,16,1
3,2015-07-04,230.0,108,23,2
4,2015-07-05,256.0,122,30,1
5,2015-07-06,304.0,142,40,1
6,2015-07-07,334.0,153,46,1
7,2015-07-08,343.0,157,48,1
8,2015-07-09,373.0,168,47,1
9,2015-07-10,395.0,177,53,5


In [113]:
# derive hotel occupancy rate
# here, we will use 95% percentile as the proxy for a full house without having a single or few outliers skewing it


FULL_HOUSE = int(daily['guests_in_hotel'].quantile(0.95))

daily['hotel_occupancy_rate'] = ((daily['guests_in_hotel']/FULL_HOUSE)* 100).clip(upper=100.0).round(2)

daily.head(100)

,date,guests_in_hotel,guests_with_breakfast,guests_half_board,guests_full_board,hotel_occupancy_rate
0,2015-07-01,68.0,36,4,1,15.89
1,2015-07-02,131.0,64,11,2,30.61
2,2015-07-03,168.0,81,16,1,39.25
3,2015-07-04,230.0,108,23,2,53.74
4,2015-07-05,256.0,122,30,1,59.81
...,...,...,...,...,...,...
95,2015-10-04,336.0,174,47,0,78.50
96,2015-10-05,346.0,178,42,6,80.84
97,2015-10-06,357.0,183,52,7,83.41
98,2015-10-07,355.0,179,50,9,82.94


In [114]:
#simulate weather information
# 0.0 -> perfect weather, 1.0 -> severe
#Hungary has cold harsh winters, warm sunny summers and wet springs/autumns


np.random.seed(42)

season_params = {
    "winter": {"months": [12, 1, 2],  "mean": 0.72, "std": 0.15},
    "spring": {"months": [3, 4, 5],   "mean": 0.40, "std": 0.18},
    "summer": {"months": [6, 7, 8],   "mean": 0.18, "std": 0.20},
    "autumn": {"months": [9, 10, 11], "mean": 0.52, "std": 0.18},
}

month_to_season = {month: season for season, info in season_params.items() for month in info["months"]}


def simulate_weather(month):
    params = season_params[month_to_season[month]]
    return float(np.clip(np.random.normal(params["mean"], params["std"]), 0, 1).round(4))
daily['month']       = daily['date'].dt.month   
daily['season'] =  daily['month'].map(month_to_season)
daily['weather_score'] = daily['month'].apply(simulate_weather)
daily.head(10)

,date,guests_in_hotel,guests_with_breakfast,guests_half_board,guests_full_board,hotel_occupancy_rate,month,season,weather_score
0,2015-07-01,68.0,36,4,1,15.89,7,summer,0.2793
1,2015-07-02,131.0,64,11,2,30.61,7,summer,0.1523
2,2015-07-03,168.0,81,16,1,39.25,7,summer,0.3095
3,2015-07-04,230.0,108,23,2,53.74,7,summer,0.4846
4,2015-07-05,256.0,122,30,1,59.81,7,summer,0.1332
5,2015-07-06,304.0,142,40,1,71.03,7,summer,0.1332
6,2015-07-07,334.0,153,46,1,78.04,7,summer,0.4958
7,2015-07-08,343.0,157,48,1,80.14,7,summer,0.3335
8,2015-07-09,373.0,168,47,1,87.15,7,summer,0.0861
9,2015-07-10,395.0,177,53,5,92.29,7,summer,0.2885


In [118]:
# hungarian, international public holidays and weekends -> flag them as special days as they increase traffic to a restaurant

daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend'] = daily['day_of_week'] >= 5

special_days = {
    (1, 1): 'New Year',
    (3, 15): 'Revolution Day',
    (5, 1): 'Labour Day',
    (8, 20): "St. Stephen's Day",
    (10, 23): 'Republic Day',
    (11, 1): 'All Saints Day',
    (12, 25): 'Christmas Day',
    (12, 26): 'Boxing Day',
    (2, 14): 'Valentines Day',
    (12, 31): "New Year's Eve"
}
# Easter is variable — add for the years in your dataset
# 2015: Apr 5, 2016: Mar 27, 2017: Apr 16
# Mother's day is also a variable - add for the years in the dataset
# 2015: May 3, 2016: May 1, 2017: May 7
easter_dates = {
    (2015, 4, 5), (2015, 4, 6), (2015, 4, 7),
    (2016, 3, 27), (2016, 3, 28),(2016, 3, 29),
    (2017, 4, 16), (2017, 4, 17),(2017, 4, 18),
}
mothers_day = {
    (2015, 5, 3),
   # (2016, 5, 1), we will leave this out since this is already a public holiday
    (2017, 5, 7),   
}
def is_holiday(date):
    if (date.year, date.month, date.day) in easter_dates:
        return True
    if (date.year, date.month, date.day) in mothers_day:
        return True
    return (date.month, date.day) in special_days
daily['is_special_day'] = daily['date'].apply(is_holiday)
daily.head(10)


,date,guests_in_hotel,guests_with_breakfast,guests_half_board,guests_full_board,hotel_occupancy_rate,month,season,weather_score,day_of_week,is_weekend,is_special_day
0,2015-07-01,68.0,36,4,1,15.89,7,summer,0.2793,2,False,False
1,2015-07-02,131.0,64,11,2,30.61,7,summer,0.1523,3,False,False
2,2015-07-03,168.0,81,16,1,39.25,7,summer,0.3095,4,False,False
3,2015-07-04,230.0,108,23,2,53.74,7,summer,0.4846,5,True,False
4,2015-07-05,256.0,122,30,1,59.81,7,summer,0.1332,6,True,False
5,2015-07-06,304.0,142,40,1,71.03,7,summer,0.1332,0,False,False
6,2015-07-07,334.0,153,46,1,78.04,7,summer,0.4958,1,False,False
7,2015-07-08,343.0,157,48,1,80.14,7,summer,0.3335,2,False,False
8,2015-07-09,373.0,168,47,1,87.15,7,summer,0.0861,3,False,False
9,2015-07-10,395.0,177,53,5,92.29,7,summer,0.2885,4,False,False


In [119]:
#simulate local events

#conferences cluster in spring/autumn, festivals/concerts/parties cluster in the summer

def simulate_event(row):
    base_probability = {
        'winter': 0.05,
        'spring': 0.15, #conference season
        'summer': 0.20, #festivals, parties, concerts
        'autumn': 0.15  #conference season
    }[row['season']]

    if row['is_weekend']: base_probability += 0.10
    if row['is_special_day']: base_probability += 0.12

    return np.random.random() < base_probability

daily['has_local_event'] = daily.apply(simulate_event, axis=1)



daily.head(10)


,date,guests_in_hotel,guests_with_breakfast,guests_half_board,guests_full_board,hotel_occupancy_rate,month,season,weather_score,day_of_week,is_weekend,is_special_day,has_local_event
0,2015-07-01,68.0,36,4,1,15.89,7,summer,0.2793,2,False,False,True
1,2015-07-02,131.0,64,11,2,30.61,7,summer,0.1523,3,False,False,False
2,2015-07-03,168.0,81,16,1,39.25,7,summer,0.3095,4,False,False,False
3,2015-07-04,230.0,108,23,2,53.74,7,summer,0.4846,5,True,False,False
4,2015-07-05,256.0,122,30,1,59.81,7,summer,0.1332,6,True,False,True
5,2015-07-06,304.0,142,40,1,71.03,7,summer,0.1332,0,False,False,False
6,2015-07-07,334.0,153,46,1,78.04,7,summer,0.4958,1,False,False,False
7,2015-07-08,343.0,157,48,1,80.14,7,summer,0.3335,2,False,False,False
8,2015-07-09,373.0,168,47,1,87.15,7,summer,0.0861,3,False,False,False
9,2015-07-10,395.0,177,53,5,92.29,7,summer,0.2885,4,False,False,False


In [120]:
# percentage of inhouse guests who are on a meal plan

daily['%breakfast'] = ((daily['guests_with_breakfast'] / daily['guests_in_hotel'])*100).round(2)
daily['%half_board'] = ((daily['guests_half_board']    / daily['guests_in_hotel'])*100).round(2)
daily['%full_board'] = ((daily['guests_full_board']    / daily['guests_in_hotel'])*100).round(2)
daily = daily.drop(columns=['guests_with_breakfast', 'guests_half_board', 'guests_full_board'])
daily.head(10)

,date,guests_in_hotel,hotel_occupancy_rate,month,season,weather_score,day_of_week,is_weekend,is_special_day,has_local_event,%breakfast,%half_board,%full_board
0,2015-07-01,68.0,15.89,7,summer,0.2793,2,False,False,True,52.94,5.88,1.47
1,2015-07-02,131.0,30.61,7,summer,0.1523,3,False,False,False,48.85,8.40,1.53
2,2015-07-03,168.0,39.25,7,summer,0.3095,4,False,False,False,48.21,9.52,0.60
3,2015-07-04,230.0,53.74,7,summer,0.4846,5,True,False,False,46.96,10.00,0.87
4,2015-07-05,256.0,59.81,7,summer,0.1332,6,True,False,True,47.66,11.72,0.39
5,2015-07-06,304.0,71.03,7,summer,0.1332,0,False,False,False,46.71,13.16,0.33
6,2015-07-07,334.0,78.04,7,summer,0.4958,1,False,False,False,45.81,13.77,0.30
7,2015-07-08,343.0,80.14,7,summer,0.3335,2,False,False,False,45.77,13.99,0.29
8,2015-07-09,373.0,87.15,7,summer,0.0861,3,False,False,False,45.04,12.60,0.27
9,2015-07-10,395.0,92.29,7,summer,0.2885,4,False,False,False,44.81,13.42,1.27


In [124]:
# upto this point, we have simulated general data that would be very useful for a morning shift, but we need information that would also be 
# useful for dinners and this is where reservation data comes from. I have worked in a restaurant since July 2024 upto now (March 2026) so I 
# have domain specific knowledge that is enough to simulate this data.


def simulate_dinner_cover_rate(row):
    month = row['date'].month
    if month in [1, 2, 3]: base = 0.10 # Q1 is usually slow in hospitality generally -> cool down period after very eventful Christmas break
    elif month in [4, 5]: base = 0.35 # momentum picks up as the weather gets better, but not quite yet
    elif month in [6, 7, 8]: base = 0.75 # summer sunny days, lots of tourists -> peak season for hospitality
    elif month in [9, 10, 11]: base = 0.45 #momentum slows down as fall breeze enters and stays fairly the same during the three months
    else: base = 0.55 # Holiday spirits, some people travel but not that much. Peaks during Christmas day, remains fairly neutral because of cold weather
    
    day_of_week = row['date'].dayofweek

    if day_of_week in [4,5,6]: base += 0.20
    elif day_of_week in [0,1,2,3]: base -= 0.15
    

    if row['is_special_day']:
        base += 0.30

    if row['has_local_event']:
        base += 0.20

    if row['weather_score'] > 0.6 and row['hotel_occupancy_rate'] > 40:
        base += 0.08

    noise = np.random.normal(0, 0.05)
    return float(np.clip(base + noise, 0, 1).round(4)) * 100

np.random.seed(42)
daily['dinner_cover_rate'] = daily.apply(simulate_dinner_cover_rate, axis=1)
daily.head(100)
    
        
    


,date,guests_in_hotel,hotel_occupancy_rate,month,season,weather_score,day_of_week,is_weekend,is_special_day,has_local_event,%breakfast,%half_board,%full_board,dinner_cover_rate
0,2015-07-01,68.0,15.89,7,summer,0.2793,2,False,False,True,52.94,5.88,1.47,82.48
1,2015-07-02,131.0,30.61,7,summer,0.1523,3,False,False,False,48.85,8.40,1.53,59.31
2,2015-07-03,168.0,39.25,7,summer,0.3095,4,False,False,False,48.21,9.52,0.60,98.24
3,2015-07-04,230.0,53.74,7,summer,0.4846,5,True,False,False,46.96,10.00,0.87,100.00
4,2015-07-05,256.0,59.81,7,summer,0.1332,6,True,False,True,47.66,11.72,0.39,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2015-10-04,336.0,78.50,10,autumn,0.2566,6,True,False,False,51.79,13.99,0.00,57.68
96,2015-10-05,346.0,80.84,10,autumn,0.5733,0,False,False,False,51.45,12.14,1.73,31.48
97,2015-10-06,357.0,83.41,10,autumn,0.5670,1,False,False,False,51.26,14.57,1.96,31.31
98,2015-10-07,355.0,82.94,10,autumn,0.5209,2,False,False,True,50.42,14.08,2.54,50.03
